# BIA-X — Ollama no Google Colab

**Banking Intelligence Assistant — Explainable Experience**

Notebook de preparação do ambiente para executar o protótipo BIA-X com Ollama em ambiente temporário do Google Colab.

> Os dados utilizados pelo projeto são sintéticos. Nenhuma operação bancária real é executada.


## Objetivo

Este notebook prepara e valida o ambiente de execução do BIA-X antes da etapa de avaliação dos cenários C01–C12.

Fluxo:

```text
Colab
  ↓
Ollama
  ↓
modelo configurado
  ↓
teste da API
  ↓
projeto BIA-X
  ↓
src/app.py
```


## 01 — Verificar o ambiente


In [ ]:
import os
import sys
import platform

print('Python:', sys.version.split()[0])
print('Sistema:', platform.system(), platform.release())
print('Ambiente Colab:', 'COLAB_RELEASE_TAG' in os.environ)


## 02 — Instalar Ollama

A instalação ocorre somente no ambiente temporário desta sessão do Colab.


In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh


## 03 — Iniciar o servidor Ollama


In [ ]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)
print('Servidor Ollama iniciado no processo:', ollama_process.pid)


## 04 — Verificar a API do Ollama


In [ ]:
import requests

OLLAMA_URL = 'http://127.0.0.1:11434'

response = requests.get(
    f'{OLLAMA_URL}/api/tags',
    timeout=20
)

response.raise_for_status()

print('Ollama respondeu corretamente.')
print(response.json())


## 05 — Configurar o modelo

O projeto utiliza `gpt-oss` como padrão. Se o modelo não estiver disponível no ambiente, a célula seguinte fará o download para a sessão atual.


In [ ]:
OLLAMA_MODEL = 'gpt-oss'

print('Modelo configurado:', OLLAMA_MODEL)
print('Observação: o download do modelo pode consumir tempo e espaço da sessão do Colab.')


In [ ]:
!ollama pull gpt-oss


## 06 — Testar o modelo diretamente


In [ ]:
payload = {
    'model': OLLAMA_MODEL,
    'prompt': 'Responda apenas: BIA-X Ollama OK',
    'stream': False
}

response = requests.post(
    f'{OLLAMA_URL}/api/generate',
    json=payload,
    timeout=120
)

response.raise_for_status()

result = response.json()
print(result.get('response', 'Resposta vazia.'))


## 07 — Configurar variáveis usadas pelo BIA-X


In [ ]:
os.environ['OLLAMA_URL'] = f'{OLLAMA_URL}/api/generate'
os.environ['OLLAMA_MODEL'] = OLLAMA_MODEL
os.environ['OLLAMA_TIMEOUT'] = '120'

print('OLLAMA_URL:', os.environ['OLLAMA_URL'])
print('OLLAMA_MODEL:', os.environ['OLLAMA_MODEL'])
print('OLLAMA_TIMEOUT:', os.environ['OLLAMA_TIMEOUT'])


## 08 — Preparar o projeto BIA-X

O notebook pode trabalhar com o repositório clonado no Colab. Se o repositório for privado, use o método de autenticação apropriado sem colocar tokens diretamente no notebook versionado.


In [ ]:
from pathlib import Path

PROJECT_DIR = Path('/content/bia-x-cognitive-bank-guardian')

print('Diretório esperado:', PROJECT_DIR)
print('Existe:', PROJECT_DIR.exists())


### Clonar o repositório

Execute somente se o projeto ainda não estiver no ambiente do Colab.


In [ ]:
# Descomente e ajuste o comando se necessário.
# !git clone https://github.com/SEU_USUARIO/bia-x-cognitive-bank-guardian.git /content/bia-x-cognitive-bank-guardian


## 09 — Verificar a Knowledge Base


In [ ]:
required_files = [
    PROJECT_DIR / 'data' / 'transactions.csv',
    PROJECT_DIR / 'data' / 'service_history.csv',
    PROJECT_DIR / 'data' / 'customer_profile.json',
    PROJECT_DIR / 'data' / 'financial_products.json',
    PROJECT_DIR / 'src' / 'app.py'
]

for path in required_files:
    print(f'{path}:', 'OK' if path.exists() else 'AUSENTE')


## 10 — Validar os dados sintéticos


In [ ]:
import json
import pandas as pd

transactions_path = PROJECT_DIR / 'data' / 'transactions.csv'
history_path = PROJECT_DIR / 'data' / 'service_history.csv'
profile_path = PROJECT_DIR / 'data' / 'customer_profile.json'
products_path = PROJECT_DIR / 'data' / 'financial_products.json'

transactions = pd.read_csv(transactions_path)
history = pd.read_csv(history_path)

with open(profile_path, encoding='utf-8') as file:
    profile = json.load(file)

with open(products_path, encoding='utf-8') as file:
    products = json.load(file)

print('Transações:', len(transactions))
print('Atendimentos:', len(history))
print('Cliente:', profile.get('cliente_id'))
print('Produtos:', len(products.get('produtos', [])))


## 11 — Verificar o `src/app.py`


In [ ]:
APP_PATH = PROJECT_DIR / 'src' / 'app.py'

if APP_PATH.exists():
    print('src/app.py encontrado.')
    print('Tamanho:', APP_PATH.stat().st_size, 'bytes')
else:
    print('src/app.py não encontrado.')


## 12 — Próxima etapa: executar o Streamlit

A execução da interface será feita depois de validar todas as etapas anteriores.

Comando esperado:

```bash
streamlit run src/app.py
```

No Colab, a forma de expor uma aplicação web pode depender do ambiente de execução. Por isso, esta etapa fica separada da validação do Ollama.


## Checklist do ambiente

- [ ] Ollama instalado na sessão do Colab
- [ ] Servidor Ollama iniciado
- [ ] API `/api/tags` respondendo
- [ ] Modelo configurado
- [ ] Modelo testado diretamente
- [ ] Variáveis `OLLAMA_URL` e `OLLAMA_MODEL` configuradas
- [ ] Projeto BIA-X disponível no Colab
- [ ] Quatro arquivos da Knowledge Base encontrados
- [ ] `src/app.py` encontrado
- [ ] Ambiente pronto para executar o BIA-X

**Este notebook prepara o ambiente. Os testes comportamentais C01–C12 serão realizados em uma etapa posterior.**
